# Fase 2 (lanjutan) — Audit Kualitas Data

Notebook ini menjalankan pengecekan duplikasi, referential integrity, dan distribusi nilai langsung di `olist_db`, sebagai bagian akhir Fase 2 sebelum lanjut ke Fase 3/4.

**Prasyarat:** `01_load_to_postgres.ipynb` sudah selesai jalan dengan semua status `OK`.

In [1]:
import os
import getpass
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text, URL
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("../.env"))

DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "olist_db")

if not DB_PASSWORD:
    DB_PASSWORD = getpass.getpass("Password PostgreSQL: ")

url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER, password=DB_PASSWORD,
    host=DB_HOST, port=int(DB_PORT), database=DB_NAME,
)
engine = create_engine(url)

def q(sql):
    return pd.read_sql(text(sql), engine)

## 1. Cek Duplikasi

> **Temuan (dari analisis CSV):** `review_id` di `order_reviews` punya **814 baris duplikat (0.82%)** — dataset asli Olist memang punya kasus ini. Kolom kunci komposit di `order_items` (`order_id`+`order_item_id`) dan `order_payments` (`order_id`+`payment_sequential`) **bersih, tidak ada duplikasi**.

In [ ]:
q("""
SELECT review_id, COUNT(*) AS jumlah
FROM order_reviews
GROUP BY review_id
HAVING COUNT(*) > 1
ORDER BY jumlah DESC;
""")